# UBER - Ride Acceptance Rates Across Geographic Zones

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [3]:
df_zone = pd.read_csv('../Data/015/fct_zone_daily_rides.csv', parse_dates=['ride_date'])

pl_zone = pl.read_csv('../Data/015/fct_zone_daily_rides.csv', try_parse_dates=True)

# Pregunta 1

### Para cada zona geográfica, ¿cuál es la tasa de aceptación mínima observada durante el segundo trimestre (Quarter 2) de 2024? Esta información ayudará a evaluar el peor desempeño de aceptación de los conductores por zona.

```SQL
SELECT
    zone_name,
    MIN(acceptance_rate) AS min_accepted_request
FROM fct_zone_daily_rides
WHERE ride_date BETWEEN '2024-04-01' AND '2024-06-30'
GROUP BY zone_name
```

In [7]:
q2 = df_zone[
    (df_zone['ride_date'].between('2024-04-01','2024-06-30'))
].groupby('zone_name').agg(
    min_accepted_rate = ('acceptance_rate', 'min')
).round(2).reset_index()

In [12]:
q2 = pl_zone.filter(
    pl.col('ride_date').is_between(date(2024,4,1),date(2024,6,30))
).group_by('zone_name').agg(
    pl.col('acceptance_rate').min().round(3).alias('min_accepted_rate')
)

# Pregunta 2

### Haz una lista de las distintas zonas geográficas que tuvieron al menos un día en el segundo trimestre (Q2) de 2024 con una tasa de aceptación inferior al 50%. Esta lista se utilizará para identificar las zonas donde los conductores suelen ser más reacios a aceptar viajes.

```SQL
SELECT DISTINCT zone_name
FROM fct_zone_daily_rides
WHERE (ride_date BETWEEN '2024-04-01' AND '2024-06-30')
  AND (acceptance_rate < 0.5);
```

In [18]:
res = df_zone[
    (df_zone['ride_date'].between('2024-04-01','2024-06-30')) &
    (df_zone['acceptance_rate'] < 0.5)
].reset_index()

res = res[['zone_name']].drop_duplicates()

In [25]:
res = pl_zone.filter(
    (pl.col('ride_date').is_between(date(2024,4,1),date(2024,6,30))) &
    (pl.col('acceptance_rate') < 0.5)
).select(
    pl.col('zone_name').unique()
).with_row_index()

# Pregunta 3

### ¿Qué zona geográfica tuvo la tasa de aceptación de viajes más baja en un solo día durante el segundo trimestre (Q2) de 2024, teniendo además al menos 10 solicitudes de viaje rechazadas ese mismo día? Recuerda que cada fila de la tabla representa los datos de un solo día en una sola región geográfica.

### Esto nos ayuda a identificar combinaciones específicas de zona-día donde la aceptación fue especialmente deficiente, para orientar mejoras focalizadas.

```SQL
SELECT
    zone_name,
    ride_date,
    ROUND(acceptance_rate::NUMERIC, 2) AS low_rate
FROM fct_zone_daily_rides
WHERE (ride_date BETWEEN '2024-04-01' AND '2024-06-30') AND
      declined_requests >= 10
ORDER BY  acceptance_rate ASC
LIMIT 1;
```

In [31]:
res = df_zone[
    (df_zone['ride_date'].between('2024-04-01','2024-06-30')) &
    (df_zone['declined_requests'] >= 10)
].nsmallest(1, 'acceptance_rate')[['zone_name','ride_date', 'acceptance_rate']]
#].sort_values('acceptance_rate').head(1)[['zone_name','ride_date', 'acceptance_rate']]

In [35]:
res = pl_zone.filter(
    (pl.col('ride_date').is_between(date(2024,4,1),date(2024,6,30))) &
    (pl.col('declined_requests') >= 10)
).sort('acceptance_rate', descending=False).head(1).select(
    ['zone_name', 'ride_date', 'acceptance_rate']
)

res

zone_name,ride_date,acceptance_rate
str,date,f64
"""Airport""",2024-06-05,0.294118
